# **1. Perkenalan Dataset**

## 📌 Nama Dataset
**Wine Quality Dataset**

---

## 📂 Sumber Dataset
Dataset berasal dari **UCI Machine Learning Repository**.
- **Link Red Wine**: https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv
- **Link White Wine**: https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv
- **Pemilik**: P. Cortez, A. Cerdeira, F. Almeida, T. Matos, J. Reis (2009)

---

## 📊 Deskripsi Dataset
Dataset Wine Quality berisi data fisikokimia dari wine merah dan putih asal Portugal (Vinho Verde).
Mencatat hasil uji laboratorium dan penilaian kualitas dari para ahli wine.

| Properti | Detail |
|---|---|
| Jumlah sampel (red) | 1.599 baris |
| Jumlah sampel (white) | 4.898 baris |
| **Total setelah digabung** | **~6.497 baris** |
| Jumlah fitur | 11 fitur fisikokimia + 1 kolom wine_type |
| Target asli | `quality` (skala 0-10) |
| **Target yang digunakan** | `quality_label` (biner: 1=baik jika quality≥7, 0=tidak) |
| Tipe data | Semua numerik |

---

## 🔬 Deskripsi Fitur

| No | Nama Fitur | Satuan | Deskripsi |
|---|---|---|---|
| 1 | `fixed acidity` | g/dm³ | Keasaman tetap (tartaric acid) |
| 2 | `volatile acidity` | g/dm³ | Keasaman volatil — tinggi → rasa cuka |
| 3 | `citric acid` | g/dm³ | Asam sitrat — memberikan kesegaran |
| 4 | `residual sugar` | g/dm³ | Sisa gula setelah fermentasi |
| 5 | `chlorides` | g/dm³ | Kandungan garam |
| 6 | `free sulfur dioxide` | mg/dm³ | SO₂ bebas — mencegah mikroba |
| 7 | `total sulfur dioxide` | mg/dm³ | Total SO₂ — pelindung wine |
| 8 | `density` | g/cm³ | Kepadatan wine |
| 9 | `pH` | — | Tingkat keasaman |
| 10 | `sulphates` | g/dm³ | Sulfat — aditif antimikroba |
| 11 | `alcohol` | % vol | Kadar alkohol |
| 12 | `wine_type` | — | 0=red, 1=white (ditambahkan) |
| 🎯 | `quality` | 0–10 | Penilaian kualitas (TARGET) |

---

## 🎯 Tujuan
Membangun **model klasifikasi biner** untuk memprediksi apakah wine berkualitas baik:
- **Kelas 1 (Good)**: quality ≥ 7
- **Kelas 0 (Not Good)**: quality < 7

---

## ✅ Alasan Pemilihan

| Kriteria | Nilai |
|---|---|
| Tabular data | ✅ Cocok untuk scikit-learn |
| Bersih & lengkap | ✅ Tidak butuh cleaning kompleks |
| Ukuran optimal | ✅ ~6.500 baris |
| Fitur numerik semua | ✅ Tidak perlu encoding |
| Klasifikasi biner | ✅ Bisa hitung ROC AUC |

# **2. Import Library**

Mengimpor seluruh pustaka Python yang dibutuhkan untuk analisis data, visualisasi, preprocessing, dan pemodelan.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Libraries berhasil diimport!')
print(f'  numpy   : {np.__version__}')
print(f'  pandas  : {pd.__version__}')
import sklearn; print(f'  sklearn : {sklearn.__version__}')

# **3. Memuat Dataset**

Dataset dimuat dari UCI ML Repository. Red wine dan white wine digabungkan, lalu ditambahkan kolom `wine_type`.
Kita periksa beberapa baris awal dan struktur data untuk memastikan data dimuat dengan benar.

In [ ]:
URL_RED   = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
URL_WHITE = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv'

print('Memuat dataset dari UCI ML Repository...')
df_red   = pd.read_csv(URL_RED,   sep=';')
df_white = pd.read_csv(URL_WHITE, sep=';')
df_red['wine_type']   = 0
df_white['wine_type'] = 1
df = pd.concat([df_red, df_white], ignore_index=True)

os.makedirs('../dataset_raw', exist_ok=True)
df.to_csv('../dataset_raw/winequality.csv', index=False)
print(f'Dataset disimpan ke ../dataset_raw/winequality.csv')
print(f'Red wine   : {len(df_red):,} baris')
print(f'White wine : {len(df_white):,} baris')
print(f'Total      : {len(df):,} baris')

In [ ]:
print('=== 5 Baris Pertama ===')
df.head()

In [ ]:
print('=== 5 Baris Terakhir ===')
df.tail()

In [ ]:
print('=== STRUKTUR DATASET ===')
print(f'Shape : {df.shape[0]:,} baris x {df.shape[1]} kolom\n')
df.info()

In [ ]:
print('=== STATISTIK DESKRIPTIF ===')
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean','std']).format('{:.3f}')

# **4. Exploratory Data Analysis (EDA)**

EDA dilakukan untuk memahami karakteristik dataset secara mendalam:
- Distribusi fitur dan target
- Deteksi missing values dan outlier
- Korelasi antar fitur
- Wawasan (insight) sebelum preprocessing

In [ ]:
# 4.1 Missing Values
print('=== CEK MISSING VALUES ===')
missing = df.isnull().sum()
pct = (missing/len(df)*100).round(2)
mv_df = pd.DataFrame({'Jumlah':missing,'Persen(%)':pct})
print(mv_df)
print(f'\nTotal missing: {missing.sum()} | Status: {"BERSIH" if missing.sum()==0 else "ADA MISSING"}')

In [ ]:
# 4.2 Duplikat
n_dup = df.duplicated().sum()
print(f'Jumlah duplikat : {n_dup} ({n_dup/len(df)*100:.2f}%)')
if n_dup > 0:
    display(df[df.duplicated()].head(3))
else:
    print('Tidak ada duplikat.')

In [ ]:
# 4.3 Distribusi Target (quality)
fig, axes = plt.subplots(1, 3, figsize=(16,4))

vc_all = df['quality'].value_counts().sort_index()
axes[0].bar(vc_all.index, vc_all.values, color='steelblue', edgecolor='white')
axes[0].set_title('Distribusi Quality (Semua Wine)')
axes[0].set_xlabel('Quality Score'); axes[0].set_ylabel('Jumlah')

vc_red = df[df['wine_type']==0]['quality'].value_counts().sort_index()
axes[1].bar(vc_red.index, vc_red.values, color='firebrick', edgecolor='white')
axes[1].set_title('Distribusi Quality (Red Wine)')

vc_white = df[df['wine_type']==1]['quality'].value_counts().sort_index()
axes[2].bar(vc_white.index, vc_white.values, color='goldenrod', edgecolor='white')
axes[2].set_title('Distribusi Quality (White Wine)')

plt.suptitle('Distribusi Skor Kualitas Wine', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_quality_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight:')
print('  - Distribusi quality menyerupai bell curve (normal)')
print('  - Mayoritas wine skor 5-6 (kualitas sedang)')
print('  - Skor 8-9 sangat jarang -> class imbalance -> perlu binarisasi')

In [ ]:
# 4.4 Distribusi Setiap Fitur
numeric_cols = df.select_dtypes(include=np.number).columns.drop('quality').tolist()
n_cols = 3
n_rows = (len(numeric_cols)+n_cols-1)//n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*3.5))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].axvline(df[col].mean(),   color='red',    linestyle='--', lw=1.5, label=f'Mean={df[col].mean():.2f}')
    axes[i].axvline(df[col].median(), color='orange', linestyle='--', lw=1.5, label=f'Median={df[col].median():.2f}')
    axes[i].set_title(col); axes[i].legend(fontsize=8)

for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Distribusi Setiap Fitur (Histogram)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight:')
print('  - residual sugar & chlorides: distribusi right-skewed -> ada outlier')
print('  - alcohol & pH: mendekati distribusi normal')

In [ ]:
# 4.5 Boxplot Outlier
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*3))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    Q1=df[col].quantile(0.25); Q3=df[col].quantile(0.75); IQR=Q3-Q1
    n_out=((df[col]<Q1-1.5*IQR)|(df[col]>Q3+1.5*IQR)).sum()
    sns.boxplot(y=df[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(f'{col} ({n_out} outlier)')

for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Boxplot Deteksi Outlier (IQR)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight: Hampir semua fitur memiliki outlier. Akan ditangani dengan IQR Capping.')

In [ ]:
# 4.6 Correlation Heatmap
plt.figure(figsize=(14,10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, annot_kws={'size':9})
plt.title('Correlation Heatmap — Wine Quality Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

corr_target = df.corr()['quality'].drop('quality').sort_values(ascending=False)
print('Korelasi fitur vs quality:')
for feat, val in corr_target.items():
    bar = '|' * int(abs(val)*30)
    print(f'  {feat:30s} {val:+.4f}  {bar}')

print('\nInsight:')
print('  - alcohol: korelasi positif tertinggi (+0.44)')
print('  - volatile acidity: korelasi negatif kuat (-0.27)')
print('  - density berkorelasi kuat dengan alcohol -> potensi multikolinearitas')

In [ ]:
# 4.7 Top 6 Fitur vs Quality
corr_target = df.corr()['quality'].drop('quality')
top6 = corr_target.abs().sort_values(ascending=False).head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(16,9))
axes = axes.flatten()

for i, feat in enumerate(top6):
    df.boxplot(column=feat, by='quality', ax=axes[i])
    axes[i].set_title(f'{feat} vs Quality')
    axes[i].set_xlabel('Quality Score')

plt.suptitle('Top 6 Fitur vs Quality Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight:')
print('  - Wine quality tinggi (8-9) cenderung alcohol lebih tinggi')
print('  - Wine quality rendah cenderung volatile acidity lebih tinggi')

# **5. Data Preprocessing**

Tahapan preprocessing:
1. Menangani Missing Values
2. Menghapus Data Duplikat
3. Deteksi dan Penanganan Outlier (IQR Capping)
4. Feature Engineering (Target Biner)
5. Feature Selection
6. Train-Test Split (80:20 stratified)
7. Normalisasi/Standarisasi (StandardScaler)

In [ ]:
# 5.1 Missing Values
print('=== STEP 1: MISSING VALUES ===')
print(f'Sebelum: {df.isnull().sum().sum()} missing')
for col in df.select_dtypes(include=np.number).columns:
    n = df[col].isnull().sum()
    if n > 0:
        med = df[col].median()
        df[col] = df[col].fillna(med)
        print(f'  "{col}": {n} nilai diisi median={med:.4f}')
print(f'Sesudah: {df.isnull().sum().sum()} missing')
print('Selesai — tidak ada missing value.')

In [ ]:
# 5.2 Duplikat
print('=== STEP 2: HAPUS DUPLIKAT ===')
nb = len(df)
df = df.drop_duplicates().reset_index(drop=True)
na = len(df)
print(f'Sebelum : {nb:,} | Sesudah : {na:,} | Dihapus : {nb-na} baris')

In [ ]:
# 5.3 Outlier Handling (IQR Capping)
print('=== STEP 3: OUTLIER HANDLING (IQR CAPPING) ===')
cols_cap = [c for c in df.select_dtypes(include=np.number).columns if c!='quality']
rpt = []
for col in cols_cap:
    Q1,Q3 = df[col].quantile([0.25,0.75])
    IQR = Q3-Q1
    lo,hi = Q1-1.5*IQR, Q3+1.5*IQR
    n_out = ((df[col]<lo)|(df[col]>hi)).sum()
    df[col] = df[col].clip(lo,hi)
    rpt.append({'Fitur':col,'N_Outlier':n_out,'Lower':round(lo,4),'Upper':round(hi,4)})
print(pd.DataFrame(rpt).set_index('Fitur').to_string())
print('\nOutlier di-clip ke batas [Lower, Upper] — data tidak berkurang.')

In [ ]:
# 5.4 Feature Engineering
print('=== STEP 4: FEATURE ENGINEERING ===')
df['quality_label'] = (df['quality']>=7).astype(int)
vc = df['quality_label'].value_counts()
print(f'Kelas 0 (Not Good): {vc.get(0,0):,}  ({vc.get(0,0)/len(df)*100:.1f}%)')
print(f'Kelas 1 (Good)    : {vc.get(1,0):,}  ({vc.get(1,0)/len(df)*100:.1f}%)')

fig, axes = plt.subplots(1,2,figsize=(10,4))
colors=['#E74C3C','#27AE60']
axes[0].bar(['Not Good (0)','Good (1)'],vc.sort_index().values,color=colors,edgecolor='white',width=0.5)
axes[0].set_title('Distribusi Target Biner'); axes[0].set_ylabel('Jumlah Sampel')
for bar,val in zip(axes[0].patches,vc.sort_index().values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,str(val),ha='center',fontweight='bold')
axes[1].pie(vc.sort_index().values,labels=['Not Good (0)','Good (1)'],autopct='%1.1f%%',colors=colors,
           startangle=90,wedgeprops={'edgecolor':'white'})
axes[1].set_title('Proporsi Kelas')
plt.suptitle('Target: quality_label',fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig('eda_target_binary.png',dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
# 5.5 Feature Selection
print('=== STEP 5: FEATURE SELECTION ===')
X = df.drop(columns=['quality','quality_label'])
y = df['quality_label']
print(f'Fitur ({X.shape[1]}): {list(X.columns)}')
print(f'Target: quality_label | Shape X: {X.shape} | Shape y: {y.shape}')

In [ ]:
# 5.6 Train-Test Split
print('=== STEP 6: TRAIN-TEST SPLIT (80:20 stratified) ===')
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'y_train: {y_train.value_counts().to_dict()}')
print(f'y_test : {y_test.value_counts().to_dict()}')
print('Stratified split: proporsi kelas terjaga di train & test.')

In [ ]:
# 5.7 Scaling (StandardScaler)
print('=== STEP 7: STANDARDSCALER ===')
print('Scaler di-fit HANYA pada X_train -> mencegah data leakage!')
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),columns=X_train.columns,index=X_train.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),     columns=X_test.columns, index=X_test.index)
print(f'\nX_train mean setelah scaling (harusnya ~0):')
print(X_train_scaled.mean().round(6).to_string())

In [ ]:
# 5.8 Simpan Dataset Preprocessing
print('=== STEP 8: SIMPAN DATASET PREPROCESSING ===')
outdir = 'dataset_preprocessing'
os.makedirs(outdir, exist_ok=True)
X_train_scaled.to_csv(f'{outdir}/X_train.csv', index=False)
X_test_scaled.to_csv( f'{outdir}/X_test.csv',  index=False)
y_train.to_csv(        f'{outdir}/y_train.csv', index=False)
y_test.to_csv(         f'{outdir}/y_test.csv',  index=False)
for f in ['X_train.csv','X_test.csv','y_train.csv','y_test.csv']:
    sz = os.path.getsize(f'{outdir}/{f}')
    print(f'  {f}  ({sz:,} bytes)')
print('\nDataset preprocessing selesai disimpan!')

---
## Ringkasan Preprocessing

| Step | Teknik | Alasan |
|---|---|---|
| Missing Values | Median Imputation | Robust terhadap outlier |
| Duplikat | drop_duplicates() | Hindari bias model |
| Outlier | IQR Capping | Data tidak berkurang |
| Feature Eng | Binarize target (≥7→1) | Konversi ke klasifikasi biner |
| Feature Select | Drop quality & label | Pisahkan fitur dari target |
| Train-Test Split | 80:20 stratified | Proporsi kelas terjaga |
| Scaling | StandardScaler (fit on train) | Cegah data leakage |

Dataset siap untuk training di **Kriteria 2**.